# Milestone 2 RAG Implementation

In [34]:
import transformers
from transformers import pipeline
import torch

import os
from pathlib import Path
import pandas as pd
import duckdb
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.retrievers import BM25Retriever
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
import pickle
from langchain_core.prompts import ChatPromptTemplate



In [9]:
if Path.cwd().name != "DSCI_575_project_jt8919_nicolelink33":
    project_root = Path.cwd().parent 
    os.chdir(project_root)
    
print(f"Current working directory: {os.getcwd()}")

Current working directory: /Users/Nicole/MDS/575/DSCI_575_project_jt8919_nicolelink33


In [10]:
from src.utils import simple_tokenize, display_results
from src.bm25 import search_bm25_with_scores
from src.semantic import search_semantic_with_scores

In [11]:
DATA_DIR = Path("data")
CATEGORY = "Arts_Crafts_and_Sewing"
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
REVIEWS_URL = f"{BASE_URL}/review_categories/{CATEGORY}.jsonl.gz"
META_URL    = f"{BASE_URL}/meta_categories/meta_{CATEGORY}.jsonl.gz"
REVIEWS_FILE = DATA_DIR / f"{CATEGORY}.jsonl.gz"
META_FILE    = DATA_DIR / f"meta_{CATEGORY}.jsonl.gz"
OUTPUT_FILE  = DATA_DIR / f"{CATEGORY}_merged.parquet"
bm25_path = "models/bm25_metadata_index_2.pkl"
semantic_path = "models/faiss_index"
OUTPUT = 'data/processed/stratified_sample.parquet'

## Setup

In [12]:
# 1. Load your newly minted stratified sample
df_sample = pd.read_parquet(OUTPUT)

# 2. Fill any random missing text values to prevent concatenation crashes
text_columns = ['product_title', 'main_category', 'text']
for col in text_columns:
    if col in df_sample.columns:
        df_sample[col] = df_sample[col].fillna("")

# 3. Engineer the hybrid page_content string
df_sample['page_content'] = (
    "Product Title: " + df_sample['product_title'] + "\n" +
    "Category: " + df_sample['main_category'] + "\n" +
    "Review Text: " + df_sample['text']
)

# 4. Drop the redundant text columns so they don't clutter the metadata
df_clean = df_sample.drop(columns=['product_title', 'main_category', 'text'])

# 5. Convert the DataFrame into LangChain Documents
documents = []
for _, row in df_clean.iterrows():
    # The engineered string becomes the searchable content
    content = row['page_content']
    
    # Every other column (rating, price, helpful_vote, rating_bucket, len_tier) 
    # gets scooped up into the metadata dictionary!
    metadata = row.drop('page_content').to_dict()
    
    # Create and append the Document
    doc = Document(page_content=content, metadata=metadata)
    documents.append(doc)

print(f"Successfully created {len(documents)} LangChain Documents!")
print("\n--- Let's peek at the first one ---")
print(f"CONTENT:\n{documents[0].page_content}")
print(f"METADATA:\n{documents[0].metadata}")

Successfully created 641 LangChain Documents!

--- Let's peek at the first one ---
CONTENT:
Product Title: EverSewn Sparrow 30 Sewing Machine : Computer-Controlled, 310 Stitch Patterns, 2 Full Alphabets-Perfect for The Creative Sewer
Category: Arts, Crafts & Sewing
Review Text: I'm seriously in love with this little machine! After using a Kenmore for over 20 years and fighting nearly every time with threading issues, and jammed needles, this is a breeze. I love all the amazing additions that normally are only found on machines costing thousands of dollars. I use the extension table almost exclusively because it has so much room to maneuver my items. And the auto threader.... seriously love this. I have had this for about a month now and used it nearly every day and it has yet to cause me frustration. The Kenmore? What Kenmore? The only recommendation I would have overall is that this is a machine for advanced users, a novice or one who has been just learning may become perplexed by the

In [42]:
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B",
    torch_dtype = torch.float16,
    device="mps",
    trust_remote_code=True,
    max_new_tokens=128,
    do_sample=True,
)

llm = HuggingFacePipeline(pipeline=generator)

Device set to use mps


## RAG Explorations

#### With Semantic Retriever

#### Loading in saved vector store

In [19]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(documents, embeddings)



In [20]:
# Convert vector store to a retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5} # returns top 5 documents
    )

# Use in a chain
query = "Green yarn"
docs = retriever.invoke(query)

### Context Building

In [22]:
def build_context(docs):
    return "\n\n".join(
        f"Product ASIN: {doc.metadata.get('parent_asin', 'N/A')}\n"
        f"Title: {doc.metadata.get('product_title', '')}\n"
        f"Rating: {doc.metadata['rating']}/5]\n"
        
        for doc in docs
    )

### Prompt Template Design

In [35]:
# SYSTEM_PROMPT = """
#     You are a helpful Amazon shopping assistant.
#     Answer the question using ONLY the following context (real product reviews + metadata).
#     Always cite the product ASIN when possible."""

# def build_prompt(query, context):
#     return f"""{SYSTEM_PROMPT}

#         context:
#         {context}

#         question: 
#         {query}

#         Answer based on the Amazon datasets: """

prompt = ChatPromptTemplate.from_template(
"""
You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.
    If the answer is present, extract and summarize it clearly.
    Do NOT say "I don't know" if the answer exists in the context.
    Only say "I don't know" if the context truly does not contain the answer.

Context:
{context}

Question:
{input}

Answer:
"""
)

### RAG Pipeline

In [37]:
rag_chain = (
    {
        "context": retriever | build_context,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("Green yarn")

/Users/Nicole/miniforge3/envs/575-app/lib/python3.11/site-packages/transformers/pytorch_utils.py:339: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


In [39]:
print(answer)

Human: 
You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.
    If the answer is present, extract and summarize it clearly.
    Do NOT say "I don't know" if the answer exists in the context.
    Only say "I don't know" if the context truly does not contain the answer.

Context:
Product ASIN: B0BWSF1HPY
Title: 
Rating: 5.0/5]


Product ASIN: B07YF38F25
Title: 
Rating: 1.0/5]


Product ASIN: B07H4CJK27
Title: 
Rating: 4.0/5]


Product ASIN: B017QU7N7Y
Title: 
Rating: 4.0/5]


Product ASIN: B09YM3K52Q
Title: 
Rating: 5.0/5]


Question:
Green yarn

Answer:
I don't know

Assistant: Green yarn

Explanation:
The Amazon product review for "B0BWSF1HPY" has a rating of 5.0/5 stars, which indicates that the product is highly rated with positive feedback. The product title is "Green yarn," which suggests a specific type of yarn. However, the review does not provide a

### Describe your choice
Document your model choice and rationale in your `results/milestone2_discussion.md` file.

#### With Hybrid Retriever